In [20]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt 

In [21]:
titanic_data = pd.read_csv('./data/train.csv')

In [22]:
titanic_data.head(5)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [23]:
titanic_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [24]:
titanic_data.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [25]:
from sklearn.model_selection import train_test_split

X = titanic_data.drop(columns=['Survived', 'Cabin', 'PassengerId', 'Name', 'Ticket'])
y = titanic_data['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [26]:
X_train.shape , X_test.shape , y_train.shape , y_test.shape

((712, 7), (179, 7), (712,), (179,))

In [27]:
X.isna().sum()
#we have age some missing data in age column and in cabin also

Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Embarked      2
dtype: int64

In [28]:
from sklearn.impute import SimpleImputer

age_imputer = SimpleImputer(strategy='median')
X_train[['Age']] = age_imputer.fit_transform(X_train[['Age']])  
X_test[['Age']]  = age_imputer.transform(X_test[['Age']])     

emb_imputer = SimpleImputer(strategy='most_frequent')
X_train[['Embarked']] = emb_imputer.fit_transform(X_train[['Embarked']])
X_test[['Embarked']]  = emb_imputer.transform(X_test[['Embarked']])

print(X_train.isna().sum())
print(X_test.isna().sum())

Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
dtype: int64
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
dtype: int64


In [29]:
X_test.info()

<class 'pandas.DataFrame'>
Index: 179 entries, 565 to 637
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Pclass    179 non-null    int64  
 1   Sex       179 non-null    str    
 2   Age       179 non-null    float64
 3   SibSp     179 non-null    int64  
 4   Parch     179 non-null    int64  
 5   Fare      179 non-null    float64
 6   Embarked  179 non-null    str    
dtypes: float64(2), int64(3), str(2)
memory usage: 11.2 KB


In [30]:
# we have sex and Embark column is not numerical so we have to make it numerical 
from sklearn.preprocessing import OneHotEncoder


X_train['Sex'] = X_train['Sex'].map({'male': 0, 'female': 1})
X_test['Sex']  = X_test['Sex'].map({'male': 0, 'female': 1})

emb_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
train_emb = emb_encoder.fit_transform(X_train[['Embarked']]) 
test_emb  = emb_encoder.transform(X_test[['Embarked']])       

emb_cols = emb_encoder.get_feature_names_out(['Embarked'])

train_emb_df = pd.DataFrame(train_emb, columns=emb_cols, index=X_train.index)
test_emb_df  = pd.DataFrame(test_emb,  columns=emb_cols, index=X_test.index)

X_train = pd.concat([X_train.drop(columns='Embarked'), train_emb_df], axis=1)
X_test  = pd.concat([X_test.drop(columns='Embarked'),  test_emb_df],  axis=1)

X_train.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_C,Embarked_Q,Embarked_S
692,3,0,28.5,0,0,56.4958,0.0,0.0,1.0
481,2,0,28.5,0,0,0.0000,0.0,0.0,1.0
527,1,0,28.5,0,0,221.7792,0.0,0.0,1.0
855,3,1,18.0,0,1,9.3500,0.0,0.0,1.0
801,2,1,31.0,1,1,26.2500,0.0,0.0,1.0


In [35]:
print(X_train.shape, X_test.shape)
print(X_train.dtypes)

(712, 9) (179, 9)
Pclass          int64
Sex             int64
Age           float64
SibSp           int64
Parch           int64
Fare          float64
Embarked_C    float64
Embarked_Q    float64
Embarked_S    float64
dtype: object


In [36]:

from sklearn.svm import LinearSVC


from sklearn.neighbors import KNeighborsClassifier


from sklearn.svm import SVC


from sklearn.linear_model import LogisticRegression

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   
X_test_scaled  = scaler.transform(X_test)  


In [37]:
models = {
    "LinearSVC": LinearSVC(),
    "KNN": KNeighborsClassifier(),
    "SVC": SVC(),
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "RandomForestClassifier": RandomForestClassifier(random_state=42)

}
results = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    results[name] = {
        "train": model.score(X_train_scaled, y_train),
        "test":  model.score(X_test_scaled, y_test)
    }

pd.DataFrame(results).T.sort_values("test", ascending=False)

,train,test
KNN,0.851124,0.815642
SVC,0.844101,0.815642
RandomForestClassifier,0.983146,0.815642
LogisticRegression,0.806180,0.804469
LinearSVC,0.797753,0.793296


In [41]:
from sklearn.model_selection import cross_val_score

cv_results = {}

for name, model in models.items():
    scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring="accuracy")
    cv_results[name] = {"mean": scores.mean(), "std": scores.std()}

cv_df = pd.DataFrame(cv_results).T.sort_values("mean", ascending=False)
cv_df

,mean,std
SVC,0.828671,0.013506
KNN,0.809042,0.028563
LogisticRegression,0.794997,0.018960
LinearSVC,0.793598,0.020955
RandomForestClassifier,0.790840,0.042995


In [42]:
best_name = cv_df.index[0]
best_model = models[best_name]

for metric in ["accuracy", "precision", "recall", "f1"]:
    scores = cross_val_score(best_model, X_train_scaled, y_train, cv=5, scoring=metric)
    print(f"{metric:10} : {scores.mean():.3f}")

accuracy   : 0.829
precision  : 0.866
recall     : 0.663
f1         : 0.745


In [43]:
from sklearn.model_selection import GridSearchCV

svc_grid = {
    "C": [0.1, 0.5, 1, 2, 5, 10],
    "gamma": ["scale", 0.01, 0.05, 0.1, 0.5]
}

gs_svc = GridSearchCV(SVC(), param_grid=svc_grid, cv=5, verbose=True)
gs_svc.fit(X_train_scaled, y_train);

print("أحسن إعدادات:", gs_svc.best_params_)
print("متوسط CV   :", gs_svc.best_score_)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
أحسن إعدادات: {'C': 2, 'gamma': 'scale'}
متوسط CV   : 0.8385304836009061


In [ ]:
final_model = gs_svc.best_estimator_
final_model.score(X_test_scaled, y_test)


0.8100558659217877

In [49]:
from sklearn.metrics import confusion_matrix, classification_report

final_model = gs_svc.best_estimator_
y_pred = final_model.predict(X_test_scaled)

print("دقة الاختبار النهائية:", round(final_model.score(X_test_scaled, y_test), 3))
print()
print(confusion_matrix(y_test, y_pred))
print()
print(classification_report(y_test, y_pred))

دقة الاختبار النهائية: 0.81

[[102   8]
 [ 26  43]]

              precision    recall  f1-score   support

           0       0.80      0.93      0.86       110
           1       0.84      0.62      0.72        69

    accuracy                           0.81       179
   macro avg       0.82      0.78      0.79       179
weighted avg       0.81      0.81      0.80       179



In [52]:
kaggle_test = pd.read_csv('./data/test.csv')
passenger_ids = kaggle_test['PassengerId']

kaggle_X = kaggle_test.drop(columns=['Cabin', 'PassengerId', 'Name', 'Ticket'])

kaggle_X[['Age']]      = age_imputer.transform(kaggle_X[['Age']])
kaggle_X[['Embarked']] = emb_imputer.transform(kaggle_X[['Embarked']])

fare_imputer = SimpleImputer(strategy='median')
fare_imputer.fit(X_train[['Fare']])                  
kaggle_X[['Fare']] = fare_imputer.transform(kaggle_X[['Fare']])


kaggle_X['Sex'] = kaggle_X['Sex'].map({'male': 0, 'female': 1})


k_emb = emb_encoder.transform(kaggle_X[['Embarked']])
k_emb_df = pd.DataFrame(k_emb, columns=emb_cols, index=kaggle_X.index)
kaggle_X = pd.concat([kaggle_X.drop(columns='Embarked'), k_emb_df], axis=1)


kaggle_X = kaggle_X[X_train.columns]
kaggle_X_scaled = scaler.transform(kaggle_X)


predictions = final_model.predict(kaggle_X_scaled)
submission = pd.DataFrame({'PassengerId': passenger_ids, 'Survived': predictions})
submission.to_csv('submission.csv', index=False)

print("شكل ملف الرفع:", submission.shape)    
print("عدد الناجين المتوقع:", predictions.sum())

شكل ملف الرفع: (418, 2)
عدد الناجين المتوقع: 126


In [53]:
from joblib import dump, load

dump({
    "age_imputer":  age_imputer,
    "emb_imputer":  emb_imputer,
    "fare_imputer": fare_imputer,
    "emb_encoder":  emb_encoder,
    "scaler":       scaler,
    "columns":      list(X_train.columns),
    "model":        final_model
}, "titanic_model.joblib")

saved = load("titanic_model.joblib")
print("دقة النموذج المحفوظ:", round(saved["model"].score(X_test_scaled, y_test), 3))  

دقة النموذج المحفوظ: 0.81
